# Smoking is bad

In [ ]:
# TODO: install all the necessary packages

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
# TODO: load the data using the pathlib library to create the path to your data

data_path = Path("data") / "smoking_data.csv"
df = pd.read_csv(data_path)

df.head()

In [ ]:
# TODO: what are the dimensions of the data frame?

df.shape

### Dimensions of the dataset
The data contains 1314 observations and 6 variables.

In [ ]:
# TODO: use .describe() on your data frame: what do you notice about the string columns?

display(df.describe())
display(df.describe(include="all"))

### About the Text Columns (Strings)
df.describe() only shows the numerical columns & leaves out string columns such as "outcome" and "smoker" where the variables are strings ("Alive", "Yes" for example).

I've also added df.describe(include="all") which includes information about the string columns as well.

In [ ]:
# TODO: do not forget to check the null values

df.isnull().sum()

### Null Values

The only variable that exhibits missing observations is salary. Salary has 700 missing observations (out of 1314 in total; so approximately 53.3%).

In [ ]:
# TODO: prepare your data frame so that you can work with

df = df.drop(columns=["Unnamed: 0"])

df["alive"] = df["outcome"].map({"Alive": 1, "Dead": 0})
df["smokes"] = df["smoker"].map({"Yes": 1, "No": 0})
df["gender"] = df["gender"].map({"male": 1, "female": 0})

df.head()

First, I dropped the column that doesn't have a name.
Second, I transformed the categorical variables into boulean values (0, 1) to facilitate the analysis. 
Whilst salary exhibits many missing values this might be handeled separately later if needed.

## Smoking analysis
Let's first check what your cousin did.

In [ ]:
# TODO: Reimplement the small code snippet described in the problem set instructions.

df.groupby(["smokes"]).agg(prob=("alive", np.mean))

The results are exactly the same as shown in the problem set instructions.

# Is smoking really good?
Now it is up to you to run the analysis further and check what could be underlying your cousings results.

### Analysing the Cousin's Interpretation

We want to analyse why smokers appear to have a higher survival probability according to this data (we know that this is probably very unlikely given real work research).

What I'll do below:
1) analyse the demographics that are present in the dataset.

### Age Distribution by Smoking Status

In [ ]:
# Age distribution of smokers and non-smokers

plt.figure(figsize=(9, 4))

sns.boxplot(
    data=df,
    x="age",
    y="smoker",
    order=["Yes", "No"],
    hue="smoker",
    hue_order=["Yes", "No"],
    palette={"Yes": "#4C78A8", "No": "#F2A65A"},
    legend=False,
    width=0.45
)

plt.title("Age Distribution by Smoking Status")
plt.xlabel("Age (years)")
plt.ylabel("")
plt.yticks([0, 1], ["Smokers", "Non-smokers"])

sns.despine()
plt.tight_layout()
plt.show()

df.groupby("smoker")["age"].agg(["mean", "median"])

What we can see here is that smokers tend to be younger than non-smokers, with a mean age of 44.68 years compared with 48.70 years (similar for the median). 

The age distribution of non-smokers is also more spread out, particularly towards older ages; above 60 years there are only non-smokers. 

This suggests that age may be a confounding variable in the relationshio between smoking and survival. Since older individuals generally have lower survival probabilities, the higher average age of non-smokers could partly explain their lower overall survival rate.

However, the age difference alone does not establish whether this explains the cousin's findings. We therefore need to compare survival probabilities within age groups.

### Survival Probability by Age Group

First, I'll create age groups of 10 years.

In [ ]:
df["age_group"] = pd.cut(
    df["age"],
    bins=[18, 30, 40, 50, 60, 70, 85],
    labels=["18–30", "31–40", "41–50", "51–60", "61–70", "71–85"],
    include_lowest=True
)

Next, I'll calculate the survival probability within each group & create the line chart.

In [ ]:
survival_by_age = (
    df.groupby(["age_group", "smoker"], observed=True)["alive"]
    .mean()
    .reset_index()
)

# Create the line chart
plt.figure(figsize=(9, 5))

sns.lineplot(
    data=survival_by_age,
    x="age_group",
    y="alive",
    hue="smoker",
    marker="o",
    palette={"Yes": "#4C78A8", "No": "#F2A65A"}
)

plt.title("Survival Probability by Age Group and Smoking Status")
plt.xlabel("Age Group (years)")
plt.ylabel("Survival Probability")
plt.ylim(0, 1)
plt.legend(title="Smoker")
sns.despine()
plt.tight_layout()
plt.show()

The graph shows that survival probability decreases substantially with age for both smokers and non-smokers. Importantly, non-smokers have higher survival probabilities in most age groups, particularly between ages 31 and 60, despite having a lower overall survival probability.

Combined with the earlier finding that smokers are younger on average, this suggests that age is a confounding variable that may explain the cousin's misleading conclusion.

### Quick Check
Because the graph shows smokers doing better in the two oldest age groups, we should check whether those groups contain relatively few smokers.

In [ ]:
df.groupby(["age_group", "smoker"], observed=True)["alive"].agg(["count", "mean"])

This confirms that non-smokers have higher survival probabilities in the four youngest age groups (18–60), with the largest difference among individuals aged 51–60 (72.2% versus 61.6%).

For ages 61–70, smokers exhibit a higher survival probability (50.0% versus 41.9%), but the smoking group contains fewer observations (68 versus 124). Among individuals aged 71–85, survival probabilities are nearly identical (9.1% versus 8.5%), although the smoker estimate is based on only 33 observations.

These findings suggest that age differences contribute to the misleading overall association between smoking and survival. The apparent advantage among older smokers should be interpreted cautiously because of smaller sample sizes.

However, the results do not establish causality, and further analysis is needed to determine whether age fully explains the original association.

### What explains the results?
I will now further investigate whether gender might explain part of the observed differences.
Second, I will also compare smokers and non-smokers after accounting for differences in their age distributions.

First, let's take a look at the survival probability ba gender and smoking status.

In [ ]:
# Calculate survival probability by gender and smoking status
survival_gender = (
    df.groupby(["gender", "smoker"])["alive"]
    .agg(["mean", "count"])
    .reset_index()
)

# Create the visualisation
plt.figure(figsize=(9, 5))

ax = sns.barplot(
    data=survival_gender,
    x="gender",
    y="mean",
    hue="smoker",
    order=[0, 1],
    hue_order=["No", "Yes"],
    palette={"No": "#F2A65A", "Yes": "#4C78A8"}
)

plt.title("Survival Probability by Gender and Smoking Status")
plt.xlabel("Gender")
plt.ylabel("Survival Probability")
plt.xticks([0, 1], ["Female", "Male"])
plt.ylim(0, 1)
plt.legend(title="Smoker")

sns.despine()
plt.tight_layout()
plt.show()

# Display sample sizes and survival probabilities
display(survival_gender)

The graph shows that smokers have higher survival probabilities for both genders. However, the difference is considerably larger for men (75.5% vs. 63.7%) than for women (76.9% vs. 73.7%).

This suggests that the relationship between smoking and survival differs by gender. However, controlling for gender alone does not reverse the cousin's findings, and age may still influence these comparisons.

Second, let's look at the age-adjusted survival probabilities

In [ ]:
# Calculate the proportion of individuals in each age group
age_weights = df["age_group"].value_counts(normalize=True)

# Calculate survival probability within each age group
survival_rates = (
    df.groupby(["age_group", "smoker"], observed=True)["alive"]
    .mean()
    .unstack()
)

# Apply the same age distribution to both groups
adjusted_survival = survival_rates.mul(age_weights, axis=0).sum()

# Compare original and adjusted survival probabilities
original_survival = df.groupby("smoker")["alive"].mean()

comparison = pd.DataFrame({
    "Original": original_survival,
    "Age-adjusted": adjusted_survival
})

display(comparison)

# Visualise the comparison
comparison.T.plot(
    kind="bar",
    figsize=(9, 5),
    color=["#F2A65A", "#4C78A8"],
    width=0.6
)

plt.title("Survival Probability Before and After Age Adjustment")
plt.xlabel("")
plt.ylabel("Survival Probability")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.legend(["Non-smokers", "Smokers"], title="Smoking Status")

sns.despine()
plt.tight_layout()
plt.show()

After adjusting for differences in age distributions, the original relationship reverses: non-smokers now have a higher survival probability (73.0%) than smokers (71.4%), compared with 68.6% and 76.1% before adjustment.

This provides strong evidence that differences in age composition explain the misleading overall association. Yet these results still do not establish a causal effect of smoking on survival.